In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
# Load the resampled data (SMOTE applied)
df_train = pd.read_csv('../data/processed/fraud_data_resampled.csv')
df_test = pd.read_csv('../data/processed/fraud_data_test.csv')

print(f"Train data shape: {df_train.shape}")
print(f"Test data shape: {df_test.shape}")

print(f"\nTrain class distribution:")
print(df_train['class'].value_counts())

print(f"\nTest class distribution:")
print(df_test['class'].value_counts())

Train data shape: (187004, 194)
Test data shape: (25830, 194)

Train class distribution:
class
0    93502
1    93502
Name: count, dtype: int64

Test class distribution:
class
0    23376
1     2454
Name: count, dtype: int64


In [3]:
# Train data
X_train = df_train.drop('class', axis=1)
y_train = df_train['class']

# Test data
X_test = df_test.drop('class', axis=1)
y_test = df_test['class']

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (187004, 193)
X_test shape: (25830, 193)


In [4]:
print("=" * 60)
print("LOGISTIC REGRESSION (BASELINE)")
print("=" * 60)

# Train model
lr = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
lr.fit(X_train, y_train)

# Predictions
y_pred_lr = lr.predict(X_test)
y_proba_lr = lr.predict_proba(X_test)[:, 1]

# Metrics
metrics_lr = {
    'accuracy': accuracy_score(y_test, y_pred_lr),
    'precision': precision_score(y_test, y_pred_lr),
    'recall': recall_score(y_test, y_pred_lr),
    'f1': f1_score(y_test, y_pred_lr),
    'roc_auc': roc_auc_score(y_test, y_proba_lr)
}

print(f"Results:")
for metric, value in metrics_lr.items():
    print(f"  {metric}: {value:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))

LOGISTIC REGRESSION (BASELINE)
Results:
  accuracy: 0.7339
  precision: 0.1965
  recall: 0.5831
  f1: 0.2940
  roc_auc: 0.7093

Confusion Matrix:
[[17525  5851]
 [ 1023  1431]]


In [5]:
print("=" * 60)
print("RANDOM FOREST")
print("=" * 60)

# Train model
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced')
rf.fit(X_train, y_train)

# Predictions
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

# Metrics
metrics_rf = {
    'accuracy': accuracy_score(y_test, y_pred_rf),
    'precision': precision_score(y_test, y_pred_rf),
    'recall': recall_score(y_test, y_pred_rf),
    'f1': f1_score(y_test, y_pred_rf),
    'roc_auc': roc_auc_score(y_test, y_proba_rf)
}

print(f"Results:")
for metric, value in metrics_rf.items():
    print(f"  {metric}: {value:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

RANDOM FOREST
Results:
  accuracy: 0.9539
  precision: 0.9444
  recall: 0.5465
  f1: 0.6923
  roc_auc: 0.7726

Confusion Matrix:
[[23297    79]
 [ 1113  1341]]


In [6]:
try:
    import xgboost as xgb
    
    print("=" * 60)
    print("XGBOOST")
    print("=" * 60)
    
    # Train model
    xgb_model = xgb.XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss'
    )
    xgb_model.fit(X_train, y_train)
    
    # Predictions
    y_pred_xgb = xgb_model.predict(X_test)
    y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]
    
    # Metrics
    metrics_xgb = {
        'accuracy': accuracy_score(y_test, y_pred_xgb),
        'precision': precision_score(y_test, y_pred_xgb),
        'recall': recall_score(y_test, y_pred_xgb),
        'f1': f1_score(y_test, y_pred_xgb),
        'roc_auc': roc_auc_score(y_test, y_proba_xgb)
    }
    
    print(f"Results:")
    for metric, value in metrics_xgb.items():
        print(f"  {metric}: {value:.4f}")
    
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred_xgb))
    
except ImportError:
    print("XGBoost not installed. Skipping...")
    metrics_xgb = None

XGBOOST
Results:
  accuracy: 0.9535
  precision: 0.9384
  recall: 0.5465
  f1: 0.6907
  roc_auc: 0.7687

Confusion Matrix:
[[23288    88]
 [ 1113  1341]]


In [7]:
# Create comparison DataFrame
comparison_data = {
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy': [metrics_lr['accuracy'], metrics_rf['accuracy']],
    'Precision': [metrics_lr['precision'], metrics_rf['precision']],
    'Recall': [metrics_lr['recall'], metrics_rf['recall']],
    'F1-Score': [metrics_lr['f1'], metrics_rf['f1']],
    'ROC-AUC': [metrics_lr['roc_auc'], metrics_rf['roc_auc']]
}

if metrics_xgb is not None:
    comparison_data['Model'].append('XGBoost')
    comparison_data['Accuracy'].append(metrics_xgb['accuracy'])
    comparison_data['Precision'].append(metrics_xgb['precision'])
    comparison_data['Recall'].append(metrics_xgb['recall'])
    comparison_data['F1-Score'].append(metrics_xgb['f1'])
    comparison_data['ROC-AUC'].append(metrics_xgb['roc_auc'])

comparison_df = pd.DataFrame(comparison_data)

print("=" * 60)
print("MODEL COMPARISON")
print("=" * 60)
print(comparison_df.round(4))

# Identify best model
best_model = comparison_df.loc[comparison_df['F1-Score'].idxmax(), 'Model']
best_f1 = comparison_df['F1-Score'].max()

print(f"\n✅ Best Model: {best_model} (F1-Score: {best_f1:.4f})")

MODEL COMPARISON
                 Model  Accuracy  Precision  Recall  F1-Score  ROC-AUC
0  Logistic Regression    0.7339     0.1965  0.5831    0.2940   0.7093
1        Random Forest    0.9539     0.9444  0.5465    0.6923   0.7726
2              XGBoost    0.9535     0.9384  0.5465    0.6907   0.7687

✅ Best Model: Random Forest (F1-Score: 0.6923)


In [8]:
print("=" * 60)
print("STRATIFIED K-FOLD CROSS-VALIDATION (k=5)")
print("=" * 60)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Logistic Regression CV
lr_cv_scores = cross_val_score(lr, X_train, y_train, cv=cv, scoring='f1')
print(f"Logistic Regression - F1 scores: {lr_cv_scores}")
print(f"Mean F1: {lr_cv_scores.mean():.4f} (+/- {lr_cv_scores.std():.4f})")

# Random Forest CV
rf_cv_scores = cross_val_score(rf, X_train, y_train, cv=cv, scoring='f1')
print(f"\nRandom Forest - F1 scores: {rf_cv_scores}")
print(f"Mean F1: {rf_cv_scores.mean():.4f} (+/- {rf_cv_scores.std():.4f})")

if metrics_xgb is not None:
    xgb_cv_scores = cross_val_score(xgb_model, X_train, y_train, cv=cv, scoring='f1')
    print(f"\nXGBoost - F1 scores: {xgb_cv_scores}")
    print(f"Mean F1: {xgb_cv_scores.mean():.4f} (+/- {xgb_cv_scores.std():.4f})")

STRATIFIED K-FOLD CROSS-VALIDATION (k=5)
Logistic Regression - F1 scores: [0.77638631 0.79438502 0.77220057 0.78932336 0.77560016]
Mean F1: 0.7816 (+/- 0.0087)

Random Forest - F1 scores: [0.95689105 0.9587909  0.95846734 0.95753303 0.95657009]
Mean F1: 0.9577 (+/- 0.0009)

XGBoost - F1 scores: [0.90052417 0.90388035 0.9025659  0.90290325 0.90211795]
Mean F1: 0.9024 (+/- 0.0011)


In [9]:
# Create comparison DataFrame
comparison_data = {
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Accuracy': [metrics_lr['accuracy'], metrics_rf['accuracy'], metrics_xgb['accuracy']],
    'Precision': [metrics_lr['precision'], metrics_rf['precision'], metrics_xgb['precision']],
    'Recall': [metrics_lr['recall'], metrics_rf['recall'], metrics_xgb['recall']],
    'F1-Score': [metrics_lr['f1'], metrics_rf['f1'], metrics_xgb['f1']],
    'ROC-AUC': [metrics_lr['roc_auc'], metrics_rf['roc_auc'], metrics_xgb['roc_auc']],
    'CV-F1 (Mean)': [lr_cv_scores.mean(), rf_cv_scores.mean(), xgb_cv_scores.mean()]
}

comparison_df = pd.DataFrame(comparison_data)

print("=" * 60)
print("FINAL MODEL COMPARISON")
print("=" * 60)
print(comparison_df.round(4))

best_model = comparison_df.loc[comparison_df['F1-Score'].idxmax(), 'Model']
best_f1 = comparison_df['F1-Score'].max()

print(f"\n✅ Best Model: {best_model} (F1-Score: {best_f1:.4f})")

FINAL MODEL COMPARISON
                 Model  Accuracy  Precision  Recall  F1-Score  ROC-AUC  \
0  Logistic Regression    0.7339     0.1965  0.5831    0.2940   0.7093   
1        Random Forest    0.9539     0.9444  0.5465    0.6923   0.7726   
2              XGBoost    0.9535     0.9384  0.5465    0.6907   0.7687   

   CV-F1 (Mean)  
0        0.7816  
1        0.9577  
2        0.9024  

✅ Best Model: Random Forest (F1-Score: 0.6923)
